In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from collections import defaultdict, Counter

sns.set_style("whitegrid")

import matplotlib

# set font size to 16
matplotlib.rcParams.update({"font.size": 16})

SEED = 4

In [2]:
n_runs = 100
# ["BENIGN", "Portmap", "NetBIOS", "LDAP", "MSSQL", "UDP", "UDPLag", "Syn"]
start_type = "Syn"

#Use the csv files again
#Load accuracy, precision, recall and f1 score
def load_dict_from_csv(filename):
    df = pd.read_csv(filename)
    nested_dict = {}
    for (run, method), group in df.groupby(['Run', 'Method']):
        nested_dict.setdefault(run, {})[method] = group['Value'].tolist()
    return nested_dict

def get_summary(loaded_dict):
    methods = list(loaded_dict[0].keys())
    summary_dict = {}
    for method in methods:
        all_runs = np.array([loaded_dict[run][method] for run in range(n_runs)]) # shape: (100, array_length)
        # Compute statistics
        mean_values = np.mean(all_runs, axis=0)
        plow_values = np.percentile(all_runs, 10, axis=0)
        phigh_values = np.percentile(all_runs, 90, axis=0)
        # Store in nested structure for easy plotting
        summary_dict[method] = {
            "mean": mean_values.tolist(),
            "plow": plow_values.tolist(),
            "phigh": phigh_values.tolist()
        }
    return summary_dict


f1score_dict_loaded = load_dict_from_csv( f"./results_dictionaries/f1score_dict_{start_type}_{n_runs}.csv")
# Mean F1 Score Values
f1score_summary = get_summary(f1score_dict_loaded)


In [3]:
# Find training set size when F1 score reaches 99% for each method
f1_99_results = {}

for method in f1score_summary.keys():
    mean_f1 = np.array(f1score_summary[method]["mean"])
    
    # Find first index where F1 score >= 0.99
    idx_99 = np.where(mean_f1 >= 0.99)[0]
    
    if len(idx_99) > 0:
        first_idx = idx_99[0]
        
        # Calculate corresponding training set size based on method type
        if len(mean_f1) == 36:  # Batch method (350 points per run / 10)
            training_size = 50 + first_idx * 10
        else:  # Single method (351 points per run)
            training_size = 50 + first_idx
        
        f1_99_results[method] = {
            "index": first_idx,
            "training_set_size": training_size,
            "f1_score": mean_f1[first_idx]
        }
    else:
        f1_99_results[method] = {
            "index": None,
            "training_set_size": None,
            "f1_score": None,
            "note": "F1 score never reached 99%"
        }

# Display results
print("=" * 80)
print("F1 Score 99% Threshold Analysis")
print("=" * 80)
for method, result in f1_99_results.items():
    print(f"\nMethod: {method}")
    if result["training_set_size"] is not None:
        print(f"  Training Set Size: {result['training_set_size']}")
        print(f"  Index: {result['index']}")
        print(f"  F1 Score at this point: {result['f1_score']:.6f}")
    else:
        print(f"  {result.get('note', 'Not found')}")

# Create a summary DataFrame
df_f1_99 = pd.DataFrame([
    {"Method": method, 
     "Training_Set_Size": result["training_set_size"],
     "F1_Score": result["f1_score"]}
    for method, result in f1_99_results.items()
])

print("\n" + "=" * 80)
print("Summary Table:")
print("=" * 80)
print(df_f1_99.to_string(index=False))

F1 Score 99% Threshold Analysis

Method: KU_batch
  Training Set Size: 300
  Index: 25
  F1 Score at this point: 0.991991

Method: KU_single
  Training Set Size: 144
  Index: 94
  F1 Score at this point: 0.990772

Method: Predictproba_batch
  Training Set Size: 250
  Index: 20
  F1 Score at this point: 0.993252

Method: Predictproba_single
  Training Set Size: 160
  Index: 110
  F1 Score at this point: 0.991025

Method: QBC
  Training Set Size: 124
  Index: 74
  F1 Score at this point: 0.991211

Method: Random
  Training Set Size: 175
  Index: 125
  F1 Score at this point: 0.990239

Summary Table:
             Method  Training_Set_Size  F1_Score
           KU_batch                300  0.991991
          KU_single                144  0.990772
 Predictproba_batch                250  0.993252
Predictproba_single                160  0.991025
                QBC                124  0.991211
             Random                175  0.990239


In [4]:
# Find maximum F1 score for all methods
max_f1_results = {}

for method in f1score_summary.keys():
    mean_f1 = np.array(f1score_summary[method]["mean"])
    max_f1 = np.max(mean_f1)
    max_idx = np.argmax(mean_f1)
    
    # Calculate corresponding training set size based on method type
    if len(mean_f1) == 35:  # Batch method
        training_size = 50 + max_idx * 10
    else:  # Single method
        training_size = 50 + max_idx
    
    max_f1_results[method] = {
        "max_f1_score": max_f1,
        "index": max_idx,
        "training_set_size": training_size
    }

# Display results
print("=" * 100)
print("Maximum F1 Score for All Methods")
print("=" * 100)
for method, result in sorted(max_f1_results.items()):
    print(f"\n{method:20} | Max F1: {result['max_f1_score']:.6f} | Training Set Size: {result['training_set_size']}")

# Create summary DataFrame
df_max_f1 = pd.DataFrame([
    {"Method": method, 
     "Max_F1_Score": result["max_f1_score"],
     "Training_Set_Size": result["training_set_size"]}
    for method, result in sorted(max_f1_results.items())
])

print("\n" + "=" * 100)
print("Summary Table:")
print("=" * 100)
print(df_max_f1.to_string(index=False))

# Find the method with the highest maximum F1 score
best_method = max(max_f1_results.items(), key=lambda x: x[1]["max_f1_score"])
print(f"\n{'='*100}")
print(f"Best Method: {best_method[0]} with Max F1 Score: {best_method[1]['max_f1_score']:.6f}")
print(f"{'='*100}")

Maximum F1 Score for All Methods

KU_batch             | Max F1: 0.999550 | Training Set Size: 82

KU_single            | Max F1: 0.999621 | Training Set Size: 257

Predictproba_batch   | Max F1: 0.999573 | Training Set Size: 82

Predictproba_single  | Max F1: 0.999621 | Training Set Size: 203

QBC                  | Max F1: 0.999614 | Training Set Size: 242

Random               | Max F1: 0.995031 | Training Set Size: 398

Summary Table:
             Method  Max_F1_Score  Training_Set_Size
           KU_batch      0.999550                 82
          KU_single      0.999621                257
 Predictproba_batch      0.999573                 82
Predictproba_single      0.999621                203
                QBC      0.999614                242
             Random      0.995031                398

Best Method: Predictproba_single with Max F1 Score: 0.999621
